<a href="https://colab.research.google.com/github/slouridorodriguez/candidates/blob/main/pipelines-etl/etl_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop 1 - ETL

Se extraerán los datos del CSV de candidatos, limpiarlos y aplicar la regla de negocio de contratación, armar el modelo estrella y cargarlo en una base de datos SQLite.


In [1]:
# Importar librerías necesarias
import pandas as pd
import sqlite3
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
# Definir rutas de los archivos en Google Drive
ruta_base = "/content/drive/My Drive/Colab Notebooks/Curso ETL/datasets/"
ruta_csv = ruta_base + "candidates.csv"
ruta_dw = ruta_base + "candidates_dw.sqlite"


## Extract

In [5]:
# Cargar el CSV de candidatos
df_candidatos = pd.read_csv(ruta_csv)

print(f"Filas: {df_candidatos.shape[0]}, Columnas: {df_candidatos.shape[1]}")
df_candidatos.head()


Filas: 50000, Columnas: 10


,First Name,Last Name,Email,Country,Application Date,Yoe,Seniority,Technology,Code Challenge Score,Technical Interview
0,Danielle,Levy,ashleysellers@gmail.com,UK,2024-11-23,6.0,Manager,Mulesoft,0,0.0
1,Angel,Ortega,gary31@yahoo.com,USA,2025-03-21,19.0,NaN,Mulesoft,9,6.0
2,Joshua,Lopez,austingarcia@hotmail.com,Ecuador,2024-10-12,14.0,NaN,Mulesoft,2,5.0
3,Jeffrey,Powers,courtney32@gmail.com,Colombia,2024-10-29,10.0,Lead,Java Backend,3,NaN
4,Jill,Robinson,stokessuzanne@gmail.com,Australia,2022-04-05,7.0,Junior,QA Manual,3,3.0


In [10]:
# Tipos de datos y nulos
df_candidatos.info()
print()
print(df_candidatos.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   First Name            50000 non-null  object 
 1   Last Name             50000 non-null  object 
 2   Email                 47491 non-null  object 
 3   Country               50000 non-null  object 
 4   Application Date      50000 non-null  object 
 5   Yoe                   47509 non-null  float64
 6   Seniority             47489 non-null  object 
 7   Technology            50000 non-null  object 
 8   Code Challenge Score  50000 non-null  int64  
 9   Technical Interview   47494 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 3.8+ MB

First Name                 0
Last Name                  0
Email                   2509
Country                    0
Application Date           0
Yoe                     2491
Seniority               2511
Technology   

In [11]:
# Filas duplicadas (mismos valores en todas las columnas)
print("Filas duplicadas:", df_candidatos.duplicated().sum())


Filas duplicadas: 0


## Transform

Limpiamos, aplicamos la regla de HIRED y armamos las tablas de dimensiones y la de hechos.


In [12]:
# Quitar las filas totalmente duplicadas
df = df_candidatos.drop_duplicates().copy()
print("Filas después de quitar duplicados:", len(df))


Filas después de quitar duplicados: 50000


In [26]:
# Convertir la fecha de aplicación a formato fecha
df['Application Date'] = pd.to_datetime(df['Application Date'])

# Seniority nulo lo dejamos como "Unknown" en vez de inventarnos un valor
df['Seniority'] = df['Seniority'].fillna('Unknown')
unknown = df.loc[df['Seniority'] == 'Unknown'].sum
print (unknown)

<bound method DataFrame.sum of       First Name Last Name                     Email  Country  \
1          Angel    Ortega          gary31@yahoo.com      USA   
2         Joshua     Lopez  austingarcia@hotmail.com  Ecuador   
33        Teresa    Oneill       haileycox@gmail.com    Spain   
41       Brandon     Evans         james68@gmail.com      USA   
52        Joseph  Cisneros                       NaN  Ecuador   
...          ...       ...                       ...      ...   
49843       Cole     Davis     sanderslisa@yahoo.com   Brazil   
49910      Kevin   Gilmore          tarale@gmail.com  Ecuador   
49932      Sarah     Reyes        wmichael@yahoo.com    Spain   
49936     Ashley     Brown                       NaN    India   
49942     Lauren   Gardner       michael65@gmail.com   Mexico   

      Application Date   Yoe Seniority                         Technology  \
1           2025-03-21  19.0   Unknown                           Mulesoft   
2           2024-10-12  14.0   Unk

In [27]:
# HIRED si Code Challenge Score Y Technical Interview son >= 7
# Si Technical Interview es nulo, no cumple la condición, así que queda como no contratado
df['is_hired'] = (df['Code Challenge Score'] >= 7) & (df['Technical Interview'] >= 7)

print("Total contratados:", df['is_hired'].sum())
print("Tasa de contratación:", round(df['is_hired'].mean() * 100, 2), "%")


Total contratados: 6599
Tasa de contratación: 13.2 %


### Dimensiones


In [28]:
# Dimensión de candidatos (nombre, apellido, correo, país)
dim_candidatos = df[['First Name', 'Last Name', 'Email', 'Country']].drop_duplicates().reset_index(drop=True)
dim_candidatos.insert(0, 'candidato_id', range(1, len(dim_candidatos) + 1))
dim_candidatos.head()


,candidato_id,First Name,Last Name,Email,Country
0,1,Danielle,Levy,ashleysellers@gmail.com,UK
1,2,Angel,Ortega,gary31@yahoo.com,USA
2,3,Joshua,Lopez,austingarcia@hotmail.com,Ecuador
3,4,Jeffrey,Powers,courtney32@gmail.com,Colombia
4,5,Jill,Robinson,stokessuzanne@gmail.com,Australia


In [29]:
# Dimensión de tecnología
dim_tecnologia = df[['Technology']].drop_duplicates().reset_index(drop=True)
dim_tecnologia.insert(0, 'tecnologia_id', range(1, len(dim_tecnologia) + 1))
dim_tecnologia


,tecnologia_id,Technology
0,1,Mulesoft
1,2,Java Backend
2,3,QA Manual
3,4,Social Media Community Management
4,5,React Frontend
5,6,Data Engineer
6,7,Client Success
7,8,DevOps
8,9,Development - CMS Backend
9,10,Python Developer


In [30]:
# Dimensión de seniority
dim_seniority = df[['Seniority']].drop_duplicates().reset_index(drop=True)
dim_seniority.insert(0, 'seniority_id', range(1, len(dim_seniority) + 1))
dim_seniority


,seniority_id,Seniority
0,1,Manager
1,2,Unknown
2,3,Lead
3,4,Junior
4,5,Trainee
5,6,Senior
6,7,Mid-Level
7,8,Intern


In [32]:
# Dimensión de fecha (una fila por cada fecha distinta de aplicación)
dim_fecha = df[['Application Date']].drop_duplicates().reset_index(drop=True)
dim_fecha.insert(0, 'fecha_id', range(1, len(dim_fecha) + 1))
dim_fecha['ano'] = dim_fecha['Application Date'].dt.year
dim_fecha['mes'] = dim_fecha['Application Date'].dt.month
dim_fecha['dia'] = dim_fecha['Application Date'].dt.day
dim_fecha.head()


,fecha_id,Application Date,ano,mes,dia
0,1,2024-11-23,2024,11,23
1,2,2025-03-21,2025,3,21
2,3,2024-10-12,2024,10,12
3,4,2024-10-29,2024,10,29
4,5,2022-04-05,2022,4,5


### Tabla de hechos

In [33]:
# Unimos el dataframe limpio con cada dimensión para traer los IDs correspondientes
fact = df.merge(dim_candidatos, on=['First Name', 'Last Name', 'Email', 'Country'], how='left')
fact = fact.merge(dim_tecnologia, on='Technology', how='left')
fact = fact.merge(dim_seniority, on='Seniority', how='left')
fact = fact.merge(dim_fecha, on='Application Date', how='left')

fact_postulaciones = fact[['candidato_id', 'fecha_id', 'tecnologia_id', 'seniority_id',
                            'Yoe', 'Code Challenge Score', 'Technical Interview', 'is_hired']]
fact_postulaciones = fact_postulaciones.rename(columns={
    'Yoe': 'yoe',
    'Code Challenge Score': 'code_challenge_score',
    'Technical Interview': 'technical_interview'
})
fact_postulaciones.insert(0, 'postulacion_id', range(1, len(fact_postulaciones) + 1))
fact_postulaciones.head()


,postulacion_id,candidato_id,fecha_id,tecnologia_id,seniority_id,yoe,code_challenge_score,technical_interview,is_hired
0,1,1,1,1,1,6.0,0,0.0,False
1,2,2,2,1,2,19.0,9,6.0,False
2,3,3,3,1,2,14.0,2,5.0,False
3,4,4,4,2,3,10.0,3,NaN,False
4,5,5,5,3,4,7.0,3,3.0,False


In [34]:
# Verificar que ningún registro se quedó sin ID de alguna dimensión
fact_postulaciones[['candidato_id', 'fecha_id', 'tecnologia_id', 'seniority_id']].isnull().sum()


,0
candidato_id,0
fecha_id,0
tecnologia_id,0
seniority_id,0


## Load

Creamos las tablas del modelo estrella en SQLite y cargamos los datos.

In [48]:
# Conectar la base de datos SQLite en Drive
conn = sqlite3.connect(ruta_dw)
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS fact_postulaciones;
DROP TABLE IF EXISTS dim_candidatos;
DROP TABLE IF EXISTS dim_tecnologia;
DROP TABLE IF EXISTS dim_seniority;
DROP TABLE IF EXISTS dim_fecha;

CREATE TABLE dim_candidatos (
    candidato_id INTEGER PRIMARY KEY,
    Nombre TEXT,
    Apellido TEXT,
    Correo TEXT,
    Pais TEXT
);

CREATE TABLE dim_tecnologia (
    tecnologia_id INTEGER PRIMARY KEY,
    Tecnologia TEXT
);

CREATE TABLE dim_seniority (
    seniority_id INTEGER PRIMARY KEY,
    Seniority TEXT
);

CREATE TABLE dim_fecha (
    fecha_id INTEGER PRIMARY KEY,
    Fecha TEXT,
    Ano INTEGER,
    Mes INTEGER,
    Dia INTEGER
);

CREATE TABLE fact_postulaciones (
    postulacion_id INTEGER PRIMARY KEY,
    candidato_id INTEGER REFERENCES dim_candidatos(candidato_id),
    fecha_id INTEGER REFERENCES dim_fecha(fecha_id),
    tecnologia_id INTEGER REFERENCES dim_tecnologia(tecnologia_id),
    seniority_id INTEGER REFERENCES dim_seniority(seniority_id),
    yoe REAL,
    code_challenge_score INTEGER,
    technical_interview REAL,
    is_hired INTEGER
);
""")

print("Tablas creadas correctamente.")


Tablas creadas correctamente.


In [49]:
# Cargar cada dimensión a la base de datos (renombrando columnas para que coincidan)
dim_candidatos.rename(columns={
    'First Name': 'Nombre', 'Last Name': 'Apellido', 'Email': 'Correo', 'Country': 'Pais'
}).to_sql('dim_candidatos', conn, if_exists='append', index=False)

dim_tecnologia.rename(columns={'Technology': 'Tecnologia'}).to_sql('dim_tecnologia', conn, if_exists='append', index=False)

dim_seniority.to_sql('dim_seniority', conn, if_exists='append', index=False)

dim_fecha['dia'] = dim_fecha['Application Date'].dt.day
dim_fecha_sql = dim_fecha.rename(columns={'Application Date': 'Fecha', 'ano': 'Ano', 'mes': 'Mes', 'dia': 'Dia'}).copy()
dim_fecha_sql['Fecha'] = dim_fecha_sql['Fecha'].dt.strftime('%Y-%m-%d')
dim_fecha_sql.to_sql('dim_fecha', conn, if_exists='append', index=False)

fact_sql = fact_postulaciones.copy()
fact_sql['is_hired'] = fact_sql['is_hired'].astype(int)
fact_sql.to_sql('fact_postulaciones', conn, if_exists='append', index=False)

conn.commit()
print("Datos cargados en la base de datos.")


Datos cargados en la base de datos.


In [50]:
# Verificar que todo quedó cargado correctamente
for tabla in ['dim_candidatos', 'dim_tecnologia', 'dim_seniority', 'dim_fecha', 'fact_postulaciones']:
    n = cursor.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(tabla, ":", n, "filas")

conn.close()


dim_candidatos : 49956 filas
dim_tecnologia : 12 filas
dim_seniority : 8 filas
dim_fecha : 1827 filas
fact_postulaciones : 50000 filas
